In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
names = ["000TJ_T", "001DS_D", "002ED_E", "003TL_T", "004TR_T", "005JQ_J", "006WB_W", "007AM_A", "008SL_S", "009EP_E", "010MR_M", "011BM_B", "012EM_E", "013ZM_Z", "014ZG_Z", "015AL_A", "016MP_M", "017PB_P", "018TL_T", "019WU_W", "020CD_C", "021AD_A", "022SG_S", "023MT_M", "024JG_J", "025AA_A", "026ED_E", "027IJ_I", "028PL_P", "029RT_R"]
#names = ["020MC_M"]
results_folder = "../ResultsStudy/"
timestamps_folder = "../Timestamps/"
outlier_percent = 15

In [ ]:
def mean_interval_difference(user_timestamps, reference_timestamps):
    user_intervals = np.diff(user_timestamps)
    reference_intervals = np.diff(reference_timestamps)
    mean_diff = np.mean(np.abs(user_intervals - reference_intervals))
    return mean_diff

def mse_intervals(user_timestamps, reference_timestamps):
    user_intervals = np.diff(user_timestamps)
    reference_intervals = np.diff(reference_timestamps)
    mse = np.mean((user_intervals - reference_intervals) ** 2)
    return mse

from scipy.spatial.distance import euclidean
from fastdtw import fastdtw
import numpy as np

def dynamic_time_warping(user_timestamps, reference_timestamps):
    # Assurez-vous que les deux entrées sont des tableaux numpy unidimensionnels
    user_timestamps = np.array(user_timestamps).reshape(-1)
    reference_timestamps = np.array(reference_timestamps).reshape(-1)
    
    # Appliquer fastdtw
    distance, path = fastdtw(user_timestamps, reference_timestamps, dist=euclidean)
    return distance






In [ ]:

def calculate_normalized_segments(timestamps):
    """Normaliser les timestamps de 0% à 100% et calculer les pourcentages pour chaque segment."""
    normalized_timestamps = (np.array(timestamps) - timestamps[0]) / (timestamps[-1] - timestamps[0]) * 100
    segment_percentages = np.diff(normalized_timestamps)
    return segment_percentages

def calculate_mean_percentage_error(user_timestamps, reference_timestamps):
    """
    Calculer l'erreur moyenne des différences de pourcentage entre les segments de user_timestamps et ref_timestamps.
    
    Parameters:
    user_timestamps (list): Liste des timestamps de l'utilisateur.
    reference_timestamps (list): Liste des timestamps de référence.
    
    Returns:
    float: Erreur moyenne des différences de pourcentage entre les segments.
    """
    
    # Normaliser et calculer les pourcentages des segments pour les deux séries de timestamps
    user_segments = calculate_normalized_segments(user_timestamps)
    reference_segments = calculate_normalized_segments(reference_timestamps)
    
    # Assurer que les deux séries ont la même longueur
    min_length = min(len(user_segments), len(reference_segments))
    user_segments = user_segments[:min_length]
    reference_segments = reference_segments[:min_length]
    
    # Calculer la différence absolue des pourcentages pour chaque segment
    differences = np.abs(user_segments - reference_segments)
    
    # Calculer l'erreur moyenne
    mean_error = np.mean(differences)
    
    return mean_error

# Utilisez cette fonction pour calculer l'erreur moyenne en passant vos séries de timestamps
# Exemple:
# user_timestamps = [0, 1.5, 3.2, 4.8, 6.0]
# reference_timestamps = [0, 1.4, 3.1, 4.7, 5.9]
# mean_error = calculate_mean_percentage_error(user_timestamps, reference_timestamps)
# print(mean_error)


In [ ]:
def plot_first_task(df, name="Balou_SN"):
    done = False
    for task in df.columns:
        if task == name and not done:
            done = True
            user_timestamps = df[task].dropna().tolist()
            ref_timestamps = read_reference_timestamps(task)
            ref_timestamps = ref_timestamps[:-1]

            # Soustraction de la première valeur de la série de timestamps de l'utilisateur
            #user_timestamps = np.array(user_timestamps) - user_timestamps[0]
            user_timestamps[0] = 0
            #user_timestamps.append(ref_timestamps[len(ref_timestamps) - 1])

            # Réglage de la taille du graphique pour plus de largeur
            plt.figure(figsize=(15, 6))  # Augmentation de la largeur

            # Tracer les lignes pour les user_timestamps et ajouter des crochets aux extrémités
            for i in range(0, len(user_timestamps)-1, 2):
                plt.plot(user_timestamps[i:i+2], [1, 1], "r-", label='Lignes Utilisateur' if i == 0 else "")
                plt.plot(user_timestamps[i], 1, "r|", markersize=10)  # Marqueur pour le début du trait
                plt.plot(user_timestamps[i+1], 1, "r|", markersize=10)  # Marqueur pour la fin du trait

            # Tracer les lignes pour les ref_timestamps et ajouter des crochets aux extrémités
            for i in range(0, len(ref_timestamps)-1, 2):
                plt.plot(ref_timestamps[i:i+2], [1.005, 1.005], "g-", label='Lignes Parfaites' if i == 0 else "")
                plt.plot(ref_timestamps[i], 1.005, "g|", markersize=10)  # Marqueur pour le début du trait
                plt.plot(ref_timestamps[i+1], 1.005, "g|", markersize=10)  # Marqueur pour la fin du trait

            plt.title(f'Timestamps pour la tâche : {task}')
            plt.xlabel('Temps (secondes)')
            plt.yticks([1, 1.005], ['Timestamps Utilisateur', 'Timestamps Parfaites'])
            plt.ylim(0.98, 1.04)  # Ajustement des limites de l'axe des y pour resserrer l'affichage
            plt.legend()
            plt.show()

            break  # Arrête après la première tâche valide

def plot_first_task2(df, name="Piaf_S"):
    done = False
    for task in df.columns:
        if task == name and not done:
            done = True
            user_timestamps = df[task].dropna().tolist()
            ref_timestamps = read_reference_timestamps(task)
            ref_timestamps = ref_timestamps[:-1]

            # Soustraction de la première valeur de chaque série de timestamps
            user_timestamps = np.array(user_timestamps) - user_timestamps[0]

            plt.figure(figsize=(10, 4))

            # Tracer les lignes pour les user_timestamps
            for i in range(0, len(user_timestamps)-1, 2):
                plt.plot(user_timestamps[i:i+2], [1, 1], "r-", label='Lignes Utilisateur' if i == 0 else "")

            # Tracer les lignes pour les ref_timestamps très proches au-dessus
            for i in range(0, len(ref_timestamps)-1, 2):
                plt.plot(ref_timestamps[i:i+2], [1.05, 1.05], "g-", label='Lignes Parfaites' if i == 0 else "")

            plt.title(f'Timestamps pour la tâche : {task}')
            plt.xlabel('Temps (secondes)')
            plt.yticks([1, 1.05], ['Timestamps Utilisateur', 'Timestamps Parfaites'])
            plt.legend()
            plt.show()

            break  # Arrête après la première tâche valide

# Exemple d'utilisation
# df est le DataFrame contenant les données d'un patient
# reference_timestamps est un dictionnaire des timestamps de référence pour chaque tâche
# plot_first_task(df, reference_timestamps)

def read_reference_timestamps(task_name):
    """Lire les timestamps de référence pour une tâche donnée."""
    with open(f"{timestamps_folder}{task_name.split('.')[0]}.txt", 'r') as file:
        return [float(line.strip().replace(',', '.')) for line in file]

def calculate_percentage_errors(user_timestamps, reference_timestamps):
    # Calcul des gaps pour les timestamps de référence et de l'utilisateur
    ref_diffs = np.diff(reference_timestamps)
    user_diffs = np.diff(user_timestamps)

    # Calcul des pourcentages : (user_diffs / ref_diffs) * 100
    percentage_diffs = (user_diffs / ref_diffs) * 1

    # Calcul de la différence absolue des pourcentages par rapport à 100%
    absolute_percentage_errors = np.abs(1 - percentage_diffs)

    return absolute_percentage_errors

def calculate_errors(user_timestamps, reference_timestamps, monophasic = True):
    """Calculer les erreurs absolues et relatives entre les timestamps utilisateur et référence.
    
    Si monophasic est True, considérer uniquement les valeurs à des positions multiples de 2.
    """

    # Suppression de la dernière valeur de reference_timestamps
    reference_timestamps = reference_timestamps[:-1]
    user_timestamps[0] = 0
    #user_timestamps.append(reference_timestamps[len(reference_timestamps) - 1])

    # Conversion des listes en tableaux Numpy pour permettre la soustraction
    user_timestamps = np.array(user_timestamps)
    reference_timestamps = np.array(reference_timestamps)

    # Si monophasic est True, ne prendre en compte que les éléments à des indices pairs
    if monophasic:
        user_timestamps = user_timestamps[::2]
        reference_timestamps = reference_timestamps[::2]
    
    # Réduction des listes à la même longueur si nécessaire
    if len(user_timestamps) != len(reference_timestamps):
        #print("Reduction: ", len(user_timestamps), " ; ", len(reference_timestamps))
        min_length = min(len(user_timestamps), len(reference_timestamps))
        user_timestamps = user_timestamps[:min_length]
        reference_timestamps = reference_timestamps[:min_length]

    # Calcul des erreurs absolues
    absolute_errors = np.abs(user_timestamps - reference_timestamps)
    
    # Calcul des erreurs relatives
    relative_errors = calculate_percentage_errors(user_timestamps, reference_timestamps)

    #calcul des ecart des taux de pourcentage
    percentage_difference_error = calculate_mean_percentage_error(user_timestamps, reference_timestamps)

     # Calcul test
    #absolute_errors = dynamic_time_warping(user_timestamps, reference_timestamps)

    return absolute_errors, relative_errors, percentage_difference_error




def exclude_10percent(data):
    """Exclure les outliers pour le calcul des erreurs."""
    return np.array(sorted(data)[int(len(data) * outlier_percent / 100) : int(len(data) * (100 - outlier_percent) / 100)])

def exclude_outliers(data):
    # Calcul de la médiane
    median = np.median(data)
    
    # Calcul des quartiles
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)
    
    # Calcul de l'IQR
    IQR = Q3 - Q1
    
    # Définition des bornes pour les outliers
    lower_bound = median - 1.5 * IQR
    upper_bound = median + 1.5 * IQR
    
    # Exclusion des outliers
    filtered_data = [x for x in data if lower_bound <= x <= upper_bound]
    
    return filtered_data

In [ ]:
# Stockage des résultats
results = {}
resultsByPatient = {}

# Traitement pour chaque patient
for name in names:
    resultsByPatient[name] = {}
    patient_folder = f"{results_folder}{name}\\"
    for csv_file in os.listdir(patient_folder):
        if csv_file.endswith(name + ".csv"):
            # Lire le fichier CSV
            df = pd.read_csv(f"{patient_folder}{csv_file}", sep=';', decimal=',', skiprows=2)
            
            # Analyse de chaque tâche
            for task in df.columns:
                if task and 'Unnamed' not in task and 'ReflexTest' not in task and 'Button' not in task:
                    user_timestamps = df[task].dropna().tolist()
                    reference_timestamps = read_reference_timestamps(task)

                    # Modification ici pour inclure le nouveau calcul
                    abs_errors, rel_errors, percentage_diff_error = calculate_errors(user_timestamps, reference_timestamps)
                    abs_errors_excluded =abs_errors
                    rel_errors_excluded = exclude_outliers(rel_errors)

                    # Calculs des moyennes
                    mean_abs_error = np.mean(abs_errors)
                    mean_rel_error = np.mean(rel_errors)
                    mean_abs_error_excl = np.mean(abs_errors_excluded)
                    mean_rel_error_excl = np.mean(rel_errors_excluded)

                    # Stockage des résultats, y compris l'erreur basée sur les pourcentages des segments
                    if task.split('.')[0] not in results:
                        results[task.split('.')[0]] = {'abs_error': [], 'rel_error': [], 'abs_error_excl': [], 'rel_error_excl': [], 'percentage_diff_error': []}
                    results[task.split('.')[0]]['abs_error'].append(mean_abs_error)
                    results[task.split('.')[0]]['rel_error'].append(mean_rel_error)
                    results[task.split('.')[0]]['abs_error_excl'].append(mean_abs_error_excl)
                    results[task.split('.')[0]]['rel_error_excl'].append(mean_rel_error_excl)
                    results[task.split('.')[0]]['percentage_diff_error'].append(percentage_diff_error)

                    if task.split('.')[0] not in resultsByPatient[name]:
                        resultsByPatient[name][task.split('.')[0]] = {'abs_error': [], 'rel_error': [], 'abs_error_excl': [], 'rel_error_excl': [], 'percentage_diff_error': []}
                    resultsByPatient[name][task.split('.')[0]]['abs_error'].append(mean_abs_error)
                    resultsByPatient[name][task.split('.')[0]]['rel_error'].append(mean_rel_error)
                    resultsByPatient[name][task.split('.')[0]]['abs_error_excl'].append(mean_abs_error_excl)
                    resultsByPatient[name][task.split('.')[0]]['rel_error_excl'].append(mean_rel_error_excl)
                    resultsByPatient[name][task.split('.')[0]]['percentage_diff_error'].append(percentage_diff_error)

print(resultsByPatient)

# Calcul des moyennes par groupe de tâches
grouped_results = {
    'mean_abs_error': [],
    'mean_rel_error': [],
    'mean_abs_error_excl': [],
    'mean_rel_error_excl': [],
    'mean_percentage_diff_error': []
}

# Tri par ordre alphabétique des tâches
sorted_tasks = sorted(results.keys())
for task in sorted_tasks:
    grouped_results['mean_abs_error'].append(np.mean(results[task]['abs_error']))
    grouped_results['mean_rel_error'].append(np.mean(results[task]['rel_error']))
    grouped_results['mean_abs_error_excl'].append(np.mean(results[task]['abs_error_excl']))
    grouped_results['mean_rel_error_excl'].append(np.mean(results[task]['rel_error_excl']))
    grouped_results['mean_percentage_diff_error'].append(np.mean(results[task]['percentage_diff_error']))


# Présentation des résultats sous forme de tableau
df_results = pd.DataFrame(grouped_results, index=sorted_tasks)
print(df_results)

# Visualisation des résultats sous forme de graphique
fig, axes = plt.subplots(3, 2, figsize=(15, 15))
fig.suptitle('Analyse des Erreurs par Tâche')

# Fonction pour ajuster les étiquettes
def adjust_labels(ax):
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor", fontsize='small')

# Ajout du nouveau graphique pour l'erreur basée sur les pourcentages des segments
axes[2, 0].bar(df_results.index, df_results['mean_percentage_diff_error'])
axes[2, 0].set_title('Erreur Moyenne de Différence de Pourcentage')
adjust_labels(axes[2, 0])

# Création des graphiques
axes[0, 0].bar(df_results.index, df_results['mean_abs_error'])
axes[0, 0].set_title('Erreur Absolue Moyenne')
adjust_labels(axes[0, 0])

axes[0, 1].bar(df_results.index, df_results['mean_rel_error'])
axes[0, 1].set_title('Erreur Relative Moyenne')
adjust_labels(axes[0, 1])

axes[1, 0].bar(df_results.index, df_results['mean_abs_error_excl'])
axes[1, 0].set_title('Erreur Absolue Moyenne (Excl. Outliers)')
adjust_labels(axes[1, 0])

axes[1, 1].bar(df_results.index, df_results['mean_rel_error_excl'])
axes[1, 1].set_title('Erreur Relative Moyenne (Excl. Outliers)')
adjust_labels(axes[1, 1])

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
# Définir les groupes de tâches
task_groupsSaved = {
    #significatif
    "AllMonotone": ["Balou_B", "Papaoutai_B", "Piaf_B", "Balou_BN", "Papaoutai_BN", "Piaf_BN", "Balou_S","Papaoutai_S", "Piaf_S", "Balou_SN","Papaoutai_SN", "Piaf_SN"],
    "AllMelodic": ["Balou_M", "Papaoutai_M", "Piaf_M", "Balou_MN", "Papaoutai_MN", "Piaf_MN", "Balou_SM","Papaoutai_SM", "Piaf_SM", "Balou_SMN","Papaoutai_SMN", "Piaf_SMN"],
    
    #significatif
    "Monotone": ["Balou_B", "Papaoutai_B", "Piaf_B", "Balou_S", "Papaoutai_S", "Piaf_S"],
    "Melodic": ["Balou_M", "Papaoutai_M", "Piaf_M", "Balou_SM", "Papaoutai_SM", "Piaf_SM"],

    #significatif
    "NMonotone": ["Balou_BN", "Papaoutai_BN", "Piaf_BN", "Balou_SN", "Papaoutai_SN", "Piaf_SN"],
    "NMelodic": ["Balou_MN", "Papaoutai_MN", "Piaf_MN", "Balou_SMN", "Papaoutai_SMN", "Piaf_SMN"],
    
    #significatif (erreur relative)
    "Silence": ["Balou_SMN", "Balou_SN", "Balou_MN", "Balou_BN", "Papaoutai_SMN", "Papaoutai_SN", "Papaoutai_MN", "Papaoutai_BN", "Piaf_SMN", "Piaf_SN", "Piaf_MN", "Piaf_BN"],
    "Auditory": ["Balou_SM", "Balou_S", "Balou_M", "Balou_B", "Papaoutai_SM", "Papaoutai_S", "Papaoutai_M", "Papaoutai_B", "Piaf_SM", "Piaf_S", "Piaf_M", "Piaf_B"],
    
    #significatif
    "AllParôle": ["Balou_SMN", "Balou_S", "Balou_SN", "Balou_SM", "Papaoutai_SMN", "Papaoutai_S", "Papaoutai_SN", "Papaoutai_SM", "Piaf_SMN", "Piaf_S", "Piaf_SN", "Piaf_SM"],
    "AllNotes": ["Balou_B", "Balou_MN", "Balou_M", "Balou_BN", "Papaoutai_B", "Papaoutai_MN", "Papaoutai_M", "Papaoutai_BN", "Piaf_B", "Piaf_MN", "Piaf_M", "Piaf_BN"],
    
    #nosignificatif
    "Parôle": ["Balou_SM", "Balou_S", "Papaoutai_SM", "Papaoutai_S", "Piaf_SM", "Piaf_S"],
    "Notes": ["Balou_M", "Balou_B", "Papaoutai_M", "Papaoutai_B", "Piaf_M", "Piaf_B"],

    #significatif
    "NParôle": ["Balou_SMN", "Balou_SN", "Papaoutai_SMN", "Papaoutai_SN", "Piaf_SMN", "Piaf_SN"],
    "NNotes": ["Balou_MN", "Balou_BN", "Papaoutai_MN", "Papaoutai_BN", "Piaf_MN", "Piaf_BN"],
    
    "AllBeat": ["Balou_B", "Papaoutai_B", "Piaf_B", "Balou_BN", "Papaoutai_BN", "Piaf_BN"],
    "Beat": ["Balou_B", "Papaoutai_B", "Piaf_B"],
    "NBeat": ["Balou_BN", "Papaoutai_BN", "Piaf_BN"],

    "AllMelody": ["Balou_M", "Papaoutai_M", "Piaf_M", "Balou_MN", "Papaoutai_MN", "Piaf_MN"],
    "Melody": ["Balou_M", "Papaoutai_M", "Piaf_M"],
    "NMelody": ["Balou_MN", "Papaoutai_MN", "Piaf_MN"],

    "AllSpeech": ["Balou_S","Papaoutai_S", "Piaf_S", "Balou_SN","Papaoutai_SN", "Piaf_SN"],
    "Speech": ["Balou_S","Papaoutai_S", "Piaf_S"],
    "NSpeech": ["Balou_SN","Papaoutai_SN", "Piaf_SN"],

    "AllSinging": ["Balou_SM","Papaoutai_SM", "Piaf_SM", "Balou_SMN","Papaoutai_SMN", "Piaf_SMN"],
    "Singing": ["Balou_SM","Papaoutai_SM", "Piaf_SM"],
    "NSinging": ["Balou_SMN","Papaoutai_SMN", "Piaf_SMN"],

    "Balou": ["Balou_SMN", "Balou_S", "Balou_SN", "Balou_SM", "Balou_B", "Balou_MN", "Balou_M", "Balou_BN"],
    "Papaoutai": ["Papaoutai_SMN", "Papaoutai_S", "Papaoutai_SN", "Papaoutai_SM", "Papaoutai_B", "Papaoutai_MN", "Papaoutai_M", "Papaoutai_BN"],
    "Piaf": ["Piaf_SMN", "Piaf_S", "Piaf_SN", "Piaf_SM", "Piaf_B", "Piaf_MN", "Piaf_M", "Piaf_BN"],
}

task_groups = {
    "AllParôle": ["Balou_SMN", "Balou_S", "Balou_SN", "Balou_SM", "Papaoutai_SMN", "Papaoutai_S", "Papaoutai_SN", "Papaoutai_SM", "Piaf_SMN", "Piaf_S", "Piaf_SN", "Piaf_SM"],
    "AllNotes": ["Balou_B", "Balou_MN", "Balou_M", "Balou_BN", "Papaoutai_B", "Papaoutai_MN", "Papaoutai_M", "Papaoutai_BN", "Piaf_B", "Piaf_MN", "Piaf_M", "Piaf_BN"],
    
    }

# Initialiser un dictionnaire pour stocker les résultats groupés, ajout de 'perc_diff_error'
grouped_results = {group: {'abs_error': [], 'rel_error': [], 'abs_error_excl': [], 'rel_error_excl': [], 'percentage_diff_error': []} for group in task_groups}

# Après avoir calculé les erreurs pour chaque tâche comme dans ton code existant...
for task, data in results.items():
    for group, tasks in task_groups.items():
        if task.split('.')[0] in tasks:
            # Supposons que 'data' contient déjà 'perc_diff_error', calculée ailleurs
            grouped_results[group]['abs_error'].extend(data['abs_error'])
            grouped_results[group]['rel_error'].extend(data['rel_error'])
            grouped_results[group]['abs_error_excl'].extend(data['abs_error_excl'])
            grouped_results[group]['rel_error_excl'].extend(data['rel_error_excl'])
            grouped_results[group]['percentage_diff_error'].extend(data['percentage_diff_error'])  # Ajout des erreurs de pourcentage

# Calculer les moyennes pour chaque groupe et chaque type d'erreur, y compris la nouvelle erreur
for group, data in grouped_results.items():
    for error_type in data:
        # Vérifier si la liste n'est pas vide et calculer la moyenne
        if data[error_type]: 
            grouped_results[group][error_type] = np.mean(data[error_type])

# Convertir les résultats groupés en DataFrame pour une présentation claire
df_grouped_results = pd.DataFrame.from_dict(grouped_results, orient='index')

# Afficher les résultats groupés
print(df_grouped_results)


In [ ]:
import matplotlib.pyplot as plt

# Configuration de base pour les graphiques


# Créer une figure et des axes pour les subplots
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(16, 12))
fig.suptitle('Moyennes des Erreurs par Groupe de Tâches', fontsize=16)

# Définir les titres pour chaque subplot
titles = ['Erreur Absolue Moyenne', 'Erreur Relative Moyenne', 'Erreur Pourcentage Moyenne', 'Erreur Relative Moyenne (Excl. Outliers)']

# Itérer sur chaque type d'erreur pour créer un graphique correspondant
for i, (ax, error_type) in enumerate(zip(axes.flat, ['abs_error', 'rel_error', 'percentage_diff_error', 'rel_error_excl'])):
    # Extraire les valeurs pour le type d'erreur actuel
    values = df_grouped_results[error_type]
    # Créer un diagramme à barres
    ax.bar(values.index, values, color='skyblue')
    # Définir le titre du subplot
    ax.set_title(titles[i])
    # Rotation des étiquettes de l'axe X pour une meilleure lisibilité
    ax.set_xticklabels(values.index, rotation=45, ha='right')
    # Ajouter des étiquettes sur les barres pour afficher les valeurs
    for p in ax.patches:
        ax.annotate(format(p.get_height(), '.2f'), (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='center', xytext=(0, 10), textcoords='offset points')

# Ajuster l'espacement pour éviter le chevauchement des étiquettes
plt.tight_layout(pad=3.0)
# Afficher les graphiques
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import shapiro, levene, ttest_ind, mannwhitneyu

metrics = ['abs_error', 'rel_error', 'percentage_diff_error', 'rel_error_excl']
metric = metrics[2]
errors_group1 = []
categoryMean= []
group1 = task_groups[list(task_groups.keys())[0]]
for name in names:
    for category in group1:
        categoryMean.append(np.mean(resultsByPatient[name][category][metric]))
    errors_group1.append(np.mean(categoryMean))
    categoryMean = []
errors_group2 = []
group2 = task_groups[list(task_groups.keys())[1]]
for name in names:
    for category in group2:
        categoryMean.append(np.mean(resultsByPatient[name][category][metric]))
    errors_group2.append(np.mean(categoryMean))
    categoryMean = []

print(len(errors_group1))
print(errors_group1)
"""
for category in group1:
    for item in results[category][metric]:
        errors_group1.append(item)
errors_group2 = []
group2 = task_groups["Beat"]
for category in group2:
    for item in results[category][metric]:
        errors_group2.append(item)
"""
# Vérification de la normalité avec le test de Shapiro-Wilk
_, p_normality_group1 = shapiro(errors_group1)
_, p_normality_group2 = shapiro(errors_group2)
print(f"Normality Group 1: p = {p_normality_group1}\nNormality Groupe 2: p = {p_normality_group2}")

# Vérification de l'homogénéité des variances avec le test de Levene
_, p_levene = levene(errors_group1, errors_group2)
print(f"Levene's Test: p = {p_levene}")

# Choix du test statistique en fonction des résultats
if p_normality_group1 > 0.05 and p_normality_group2 > 0.05 and p_levene > 0.05:
    # Les données suivent une distribution normale et les variances sont homogènes
    # Utiliser le test t de Student
    _, p_ttest = ttest_ind(errors_group1, errors_group2)
    print(f"t-test: p = {p_ttest}")
else:
    # Les données ne suivent pas une distribution normale ou les variances ne sont pas homogènes
    # Utiliser le test de Mann-Whitney
    _, p_mannwhitney = mannwhitneyu(errors_group1, errors_group2)
    print(f"Mann-Whitney Test: p = {p_mannwhitney}")
